<a href="https://colab.research.google.com/github/GODRATAN/FlyRank_ML-Intern/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GODRATAN/FlyRank_ML_Assignment/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method choice

I will use **Logistic Regression** as the first learned model.

This lane is a ranking problem: the goal is to decide which content observations should be reviewed first. I will use the model's predicted probability as the ranking score and evaluate it with Precision@20 and Precision@50, matching the ranking-oriented evaluation used for the baseline.

Logistic Regression is a simple and interpretable starting point. It provides a clear learned baseline before considering a more complex model.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design

I will use a time-aware feature/target design.

- **Feature window:** March 2026.
- **Future outcome window:** April 2026.
- **Target:** a future-decline label equal to 1 when the average daily GSC impressions in April are at least 20% lower than the March average daily GSC impressions.
- **Eligibility:** keep content with at least 100 March GSC impressions and at least 14 observed GSC days in both months.
- **Validation:** hold out whole clients for testing rather than randomly splitting rows.

The 20% decline threshold is a declared decision rule for this experiment, not a universal definition of decline.

This design keeps April outcome information out of the March feature window.

In [9]:
# ML-08 — Build March features + April future-decline label

import os
import numpy as np
import pandas as pd
from google.colab import userdata
from datasets import load_dataset

HF_TOKEN = userdata.get("HF_TOKEN")

BASE = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance"

needed_cols = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_data_available",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
]

def load_month(month):
    ds = load_dataset(
        "FlyRank/internship-warehouse",
        "fact_content_daily_performance",
        data_files={
            "train": f"{BASE}/month={month}/data_0.parquet"
        },
        token=HF_TOKEN,
    )["train"]

    return ds.select_columns(needed_cols).to_pandas()


# ---------- March: feature window ----------
march_raw = load_month("2026-03")

# Only use rows where GSC data is actually available.
march_raw = march_raw[march_raw["gsc_data_available"] == True].copy()

# Treat position 0 as unavailable rather than a real ranking.
march_raw["gsc_avg_position"] = march_raw["gsc_avg_position"].replace(0, np.nan)

march_agg = (
    march_raw
    .groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        march_impressions=("gsc_impressions", "sum"),
        march_clicks=("gsc_clicks", "sum"),
        march_avg_position=("gsc_avg_position", "mean"),
        march_days_observed=("report_date", "nunique"),
    )
)

del march_raw


# ---------- April: future outcome window ----------
april_raw = load_month("2026-04")

april_raw = april_raw[april_raw["gsc_data_available"] == True].copy()

april_agg = (
    april_raw
    .groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        april_impressions=("gsc_impressions", "sum"),
        april_days_observed=("report_date", "nunique"),
    )
)

del april_raw


# ---------- Join feature window to future outcome ----------
model_data = march_agg.merge(
    april_agg,
    on=["client_hash_id", "content_hash_id"],
    how="inner",
)

# Compare average daily impressions because March has 31 days
# and April has 30 days.
model_data["march_daily_impressions"] = (
    model_data["march_impressions"] /
    model_data["march_days_observed"]
)

model_data["april_daily_impressions"] = (
    model_data["april_impressions"] /
    model_data["april_days_observed"]
)

# Only keep rows with enough observed history.
eligible = (
    (model_data["march_impressions"] >= 100)
    & (model_data["march_days_observed"] >= 14)
    & (model_data["april_days_observed"] >= 14)
)

model_data = model_data[eligible].copy()

# Future label: at least a 20% decline in average daily impressions.
model_data["future_decline_label"] = (
    model_data["april_daily_impressions"]
    <= 0.80 * model_data["march_daily_impressions"]
).astype(int)

# Safe March-only model features.
model_data["march_ctr"] = (
    model_data["march_clicks"] /
    model_data["march_impressions"].replace(0, np.nan)
)

feature_cols = [
    "march_impressions",
    "march_clicks",
    "march_ctr",
    "march_avg_position",
    "march_days_observed",
]

print("Model rows:", len(model_data))
print("Clients:", model_data["client_hash_id"].nunique())
print("Positive labels:", model_data["future_decline_label"].sum())
print(
    "Base rate:",
    round(model_data["future_decline_label"].mean(), 4)
)
print("Features:", feature_cols)

Model rows: 96268
Clients: 40
Positive labels: 48873
Base rate: 0.5077
Features: ['march_impressions', 'march_clicks', 'march_ctr', 'march_avg_position', 'march_days_observed']


In [10]:
# ML-08 — Client-grouped train/test split

from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        model_data,
        groups=model_data["client_hash_id"]
    )
)

train_data = model_data.iloc[train_idx].copy()
test_data = model_data.iloc[test_idx].copy()

train_clients = set(train_data["client_hash_id"])
test_clients = set(test_data["client_hash_id"])

print("Train rows:", len(train_data))
print("Test rows:", len(test_data))
print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(train_clients & test_clients))

print("\nTrain base rate:",
      round(train_data["future_decline_label"].mean(), 4))

print("Test base rate:",
      round(test_data["future_decline_label"].mean(), 4))

Train rows: 73857
Test rows: 22411
Train clients: 32
Test clients: 8
Client overlap: 0

Train base rate: 0.4664
Test base rate: 0.6437


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [11]:
# ML-08 — Train Logistic Regression and compare with the Week-4 baseline

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# -----------------------------
# Features and target
# -----------------------------
X_train = train_data[feature_cols]
y_train = train_data["future_decline_label"]

X_test = test_data[feature_cols]
y_test = test_data["future_decline_label"]

# -----------------------------
# Logistic Regression
# -----------------------------
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

model_scores = model.predict_proba(X_test)[:, 1]


# -----------------------------
# Week-4 baseline
# -----------------------------
# Recreate the same rule logic on the held-out test data.
# The position-specific CTR reference is calculated from TRAIN only.

position_bins = [-0.01, 3, 10, 20, 50, float("inf")]
position_labels = ["1-3", "4-10", "11-20", "21-50", "50+"]

train_tmp = train_data.copy()
test_tmp = test_data.copy()

train_tmp["position_bucket"] = pd.cut(
    train_tmp["march_avg_position"],
    bins=position_bins,
    labels=position_labels
)

test_tmp["position_bucket"] = pd.cut(
    test_tmp["march_avg_position"],
    bins=position_bins,
    labels=position_labels
)

position_ctr_reference = (
    train_tmp
    .groupby("position_bucket", observed=False)["march_ctr"]
    .mean()
)

# Convert the categorical bucket to strings and map numeric CTR values.
reference_ctr_map = {
    str(k): float(v)
    for k, v in position_ctr_reference.items()
    if pd.notna(v)
}

test_tmp["reference_ctr"] = (
    test_tmp["position_bucket"]
    .astype(str)
    .map(reference_ctr_map)
)

visible = test_tmp["march_impressions"] >= 100
low_ctr = (
    test_tmp["march_ctr"]
    < test_tmp["reference_ctr"]
)

baseline_scores = (
    visible.astype(int)
    * low_ctr.astype(int)
    * test_tmp["march_impressions"]
)


# -----------------------------
# Precision@K
# -----------------------------
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    top_k = np.asarray(labels)[order[:k]]
    return top_k.mean()


results = pd.DataFrame({
    "Method": [
        "Week-4 baseline",
        "Logistic Regression"
    ],
    "Precision@20": [
        precision_at_k(baseline_scores, y_test, 20),
        precision_at_k(model_scores, y_test, 20)
    ],
    "Precision@50": [
        precision_at_k(baseline_scores, y_test, 50),
        precision_at_k(model_scores, y_test, 50)
    ]
})

print("Test base rate:", round(y_test.mean(), 4))
print()
print(results.to_string(index=False))

Test base rate: 0.6437

             Method  Precision@20  Precision@50
    Week-4 baseline          0.50          0.52
Logistic Regression          0.85          0.82


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [12]:
# ML-08 — Errors and interpretation

# -----------------------------------------
# 1. Feature interpretation
# -----------------------------------------

logistic_model = model.named_steps["logistic"]

coefficients = pd.Series(
    logistic_model.coef_[0],
    index=feature_cols
)

feature_importance = (
    coefficients
    .abs()
    .sort_values(ascending=False)
)

print("Top features by absolute Logistic Regression coefficient:")
print(feature_importance.to_string())

print("\nSigned coefficients:")
print(
    coefficients
    .sort_values(key=lambda x: x.abs(), ascending=False)
    .to_string()
)


# -----------------------------------------
# 2. Three concrete wrong cases
# -----------------------------------------

error_cases = test_data[
    [
        "client_hash_id",
        "content_hash_id",
        "march_impressions",
        "march_clicks",
        "march_ctr",
        "march_avg_position",
        "march_days_observed",
        "future_decline_label",
    ]
].copy()

error_cases["predicted_probability"] = model_scores
error_cases["predicted_label"] = (
    error_cases["predicted_probability"] >= 0.5
).astype(int)

errors = error_cases[
    error_cases["predicted_label"]
    != error_cases["future_decline_label"]
].copy()

print("\nTotal test errors:", len(errors))

print("\nThree concrete wrong cases:")
print(
    errors[
        [
            "client_hash_id",
            "content_hash_id",
            "march_impressions",
            "march_clicks",
            "march_ctr",
            "march_avg_position",
            "future_decline_label",
            "predicted_probability",
            "predicted_label",
        ]
    ]
    .head(3)
    .to_string(index=False)
)

Top features by absolute Logistic Regression coefficient:
march_clicks           0.361931
march_ctr              0.276261
march_days_observed    0.078198
march_avg_position     0.059389
march_impressions      0.033491

Signed coefficients:
march_clicks          -0.361931
march_ctr             -0.276261
march_days_observed    0.078198
march_avg_position    -0.059389
march_impressions      0.033491

Total test errors: 9062

Three concrete wrong cases:
         client_hash_id          content_hash_id  march_impressions  march_clicks  march_ctr  march_avg_position  future_decline_label  predicted_probability  predicted_label
client_0fa64a184f18a4a0 content_0270d6efccf55d1c               3297            15   0.004550            2.385191                     1               0.356993                0
client_0fa64a184f18a4a0 content_02c2b502e0f6b42c               2072            16   0.007722            2.160126                     1               0.307549                0
client_0fa64a184f18a4

### Model interpretation

The Logistic Regression model relied most on `march_clicks` and `march_ctr`, followed by `march_days_observed`, `march_avg_position`, and `march_impressions`. Because the features were standardized before fitting, the absolute coefficient sizes provide a useful comparison of their relative influence within this model.

The negative coefficients for `march_clicks` and `march_ctr` mean that higher current search clicks and CTR were associated with a lower predicted probability of future decline in this fitted sample. This is directionally plausible because stronger current engagement can coincide with more stable performance, but it does not establish causality.

### Error analysis

The three inspected errors were false negatives: the observed March signals led the model to predict no future decline, but the April outcome met the future-decline label definition.

1. **Case 1:** 3,297 March impressions, 15 clicks, CTR 0.00455, average position 2.39. The model assigned a probability of 0.357 but the future label was 1. The page had strong current visibility and a good observed position, so the March snapshot did not clearly reveal the later decline.

2. **Case 2:** 2,072 March impressions, 16 clicks, CTR 0.00772, average position 2.16. The model assigned a probability of 0.308 while the future label was 1. Strong current CTR and position made this a difficult future-decline case for the model.

3. **Case 3:** 781 March impressions, 5 clicks, CTR 0.00640, average position 2.46. The model assigned a probability of 0.352 while the future label was 1. Again, the current March signals suggested relatively strong performance even though the following month met the decline definition.

These errors show that current-period search performance does not fully capture later changes in performance. The model can therefore miss future declines that are not visible in the March snapshot.

### Validation note

The held-out test clients had a higher positive-label base rate (0.6437) than the training clients (0.4664). This indicates that the client holdout is not distributionally identical to training data, so the reported test precision should be interpreted as performance on these held-out clients rather than as a universal estimate.

## Self-check

Before you submit, confirm each line honestly:

- [Yes] Every section above is filled — markdown thinking AND the code that backs it
- [Yes] The notebook runs top to bottom with no errors (Runtime → Run all)
- [Yes] No client names, URLs, or private queries anywhere
- [Yes] My claims use careful words: observed, measured, directional, decision-support
- [Yes] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.